# 作业2：列表，字典和类

- **学号**：20244081108
- **姓名**：王亨泰
- **环境**：独立 conda 虚拟环境 `pyda`（Python 3.11 + Jupyter）

## 第一个代码框：身份与环境记录

In [1]:
import hashlib
import os
import platform
import uuid

# 修改为自己的学号
STUDENT_ID = 20244081108

CONDA_ENV = os.environ.get("CONDA_DEFAULT_ENV", "base")
print(f"[Conda 环境] {CONDA_ENV}")

MACHINE_FP = hashlib.sha256(
    f"{platform.node()}|{platform.machine()}|{platform.system()}|{uuid.getnode()}".encode()
).hexdigest()[:16]

RUN_ID = f"{STUDENT_ID}-{MACHINE_FP}"
print(f"[学号]       {STUDENT_ID}")
print(f"[机器指纹] {MACHINE_FP}")
print(f"[RUN_ID]   {RUN_ID}")

[Conda 环境] pyda
[学号]       20244081108
[机器指纹] e8b54e0f27d3d000
[RUN_ID]   20244081108-e8b54e0f27d3d000


---
## 字符串的乐趣

### 1. is_palindrome

回文是指正着读和倒着读相同的词或短语。判断时忽略空格、标点和大小写，因此 `tacocat` 和 `T A C O cat` 都应返回 `True`。

In [2]:
def is_palindrome(s):
    cleaned = "".join(ch.lower() for ch in s if ch.isalnum())
    return cleaned == cleaned[::-1]


cases = [
    "level",
    "kayak",
    "pop",
    "rats live on no evil star",
    "Was it a car or a cat I saw?",
    "tacocat",
    "T A C O cat",
    "python",
]

for s in cases:
    print(f"{s!r:40} -> {is_palindrome(s)}")

'level'                                  -> True
'kayak'                                  -> True
'pop'                                    -> True
'rats live on no evil star'              -> True
'Was it a car or a cat I saw?'           -> True
'tacocat'                                -> True
'T A C O cat'                            -> True
'python'                                 -> False


**说明**：先把字符串转成小写，只保留字母和数字，得到 `cleaned`；再用切片 `[::-1]` 反转并比较。空格和标点在这一步被过滤掉，所以带标点的句子也能正确判断。

### 2. is_abecedarian

如果单词的字母从左到右按字母顺序排列（允许重复），则返回 `True`。例如 `adder`、`beet`、`abcd efgh xyz` 为 `True`，`dog`、`cat` 为 `False`。

In [3]:
def is_abecedarian(s):
    letters = [ch.lower() for ch in s if ch != " "]
    return all(letters[i] <= letters[i + 1] for i in range(len(letters) - 1))


for s in ["adder", "beet", "dog", "cat", "abcd efgh xyz", "a", ""]:
    print(f"{s!r:20} -> {is_abecedarian(s)}")

'adder'              -> True
'beet'               -> True
'dog'                -> False
'cat'                -> False
'abcd efgh xyz'      -> True
'a'                  -> True
''                   -> True


**说明**：去掉空格并转小写后，逐一比较相邻字母；只要出现一次“前一个字母大于后一个字母”，就返回 `False`。空字符串和单字母字符串没有相邻字母对，按定义返回 `True`。

### 3. remove_vowels

去掉字符串中的所有元音 `a、e、i、o、u`，返回新字符串。例如 `remove_vowels('cat')` 返回 `'ct'`，`remove_vowels('audio')` 返回 `'d'`。

In [4]:
def remove_vowels(s):
    vowels = "aeiouAEIOU"
    return "".join(ch for ch in s if ch not in vowels)


for s in ["cat", "audio", "aeiou", "rhythm", "Hello World"]:
    print(f"{s!r:15} -> {remove_vowels(s)!r}")

'cat'           -> 'ct'
'audio'         -> 'd'
'aeiou'         -> ''
'rhythm'        -> 'rhythm'
'Hello World'   -> 'Hll Wrld'


**说明**：用生成器表达式逐个检查字符，只保留不在元音集合中的字符；大小写元音都处理，因此 `Hello World` 中的 `e`、`o` 也会被去掉。

---
## 列表的乐趣

### 1. list_reverse

接受一个列表并返回反转后的列表；如果参数不是列表，应抛出 `TypeError`。

In [5]:
def list_reverse(lst):
    if not isinstance(lst, list):
        raise TypeError("list_reverse() 的参数必须是列表")
    return lst[::-1]


print(list_reverse([1, 2, 3]))
print(list_reverse(["a", "b", "c"]))
print(list_reverse([]))

try:
    list_reverse((1, 2, 3))
except TypeError as e:
    print("TypeError:", e)

[3, 2, 1]
['c', 'b', 'a']
[]
TypeError: list_reverse() 的参数必须是列表


**说明**：`lst[::-1]` 返回一个新列表，不修改原列表。先用 `isinstance` 检查类型，非列表输入会抛出带说明的 `TypeError`。

### 2. is_sorted

如果序列按升序排列则返回 `True`，否则返回 `False`。要求只遍历序列一次，不能使用需要多次遍历的低效写法。

In [6]:
def is_sorted(seq):
    for i in range(len(seq) - 1):
        if seq[i] > seq[i + 1]:
            return False
    return True


print(is_sorted([1, 2, 3]))
print(is_sorted([1, 3, 2]))
print(is_sorted("abc"))
print(is_sorted([2, 2, 3]))
print(is_sorted([3, 2, 1]))

True
False
True
True
False


**说明**：只用一个 `for` 循环比较相邻元素，发现逆序立即返回，因此最多遍历一次。函数不假设 `seq` 的具体类型，只要支持索引和元素比较即可，列表、字符串、元组都能直接使用。

---
## 作为向量的元组

用元组表示向量：长度为 $d$ 的元组表示 $d$ 维向量。

### 1. vec_scalar_mult

接受元组 `t` 和标量 `s`，返回每个元素乘以 `s` 后的新元组；输入类型不合适时抛出 `TypeError`。

In [7]:
def vec_scalar_mult(t, s):
    if not isinstance(t, tuple):
        raise TypeError("t 必须是元组")
    if not isinstance(s, (int, float)):
        raise TypeError("s 必须是 int 或 float")
    for v in t:
        if not isinstance(v, (int, float)):
            raise TypeError("t 的元素必须是 int 或 float")
    return tuple(v * s for v in t)


print(vec_scalar_mult((1, 2, 3), 2))
print(vec_scalar_mult((1.5, -2.0), 0.5))

try:
    vec_scalar_mult([1, 2], 2)
except TypeError as e:
    print("TypeError:", e)

try:
    vec_scalar_mult((1, 2), "2")
except TypeError as e:
    print("TypeError:", e)

(2, 4, 6)
(0.75, -1.0)
TypeError: t 必须是元组
TypeError: s 必须是 int 或 float


**说明**：整数标量和整数元组都是有效输入；检查分别覆盖了元组本身、标量和元组元素三种情况，错误信息说明了哪一部分不合适。

### 2. vec_inner_product

接受两个元组，返回内积 $\sum_{j=1}^{d} x_j y_j$（浮点数）。任一输入不是元组时抛 `TypeError`，长度不一致时抛 `ValueError`。

In [8]:
def vec_inner_product(x, y):
    if not isinstance(x, tuple) or not isinstance(y, tuple):
        raise TypeError("x 和 y 必须是元组")
    if len(x) != len(y):
        raise ValueError("两个向量长度必须相同")
    total = 0.0
    for a, b in zip(x, y):
        if not isinstance(a, (int, float)) or not isinstance(b, (int, float)):
            raise TypeError("向量元素必须是 int 或 float")
        total += a * b
    return float(total)


print(vec_inner_product((1, 2, 3), (4, 5, 6)))
print(vec_inner_product((1.5, 2.0), (0.5, 4.0)))

try:
    vec_inner_product([1, 2], (3, 4))
except TypeError as e:
    print("TypeError:", e)

try:
    vec_inner_product((1, 2), (3, 4, 5))
except ValueError as e:
    print("ValueError:", e)

32.0
8.75
TypeError: x 和 y 必须是元组
ValueError: 两个向量长度必须相同


**说明**：内积用 `zip` 逐对相乘再累加；即使输入都是整数，返回值也显式转为 `float`，与题目要求的“返回浮点数”一致。

### 3. check_valid_mx

用元组的元组表示 $m\times n$ 矩阵。有效矩阵满足：外层元组的每个元素都是元组；所有内层元组长度相同；每个元素都是整数或浮点数。

In [9]:
def check_valid_mx(my_mx):
    if not isinstance(my_mx, tuple):
        return False
    for row in my_mx:
        if not isinstance(row, tuple):
            return False
    if len(my_mx) > 0:
        n = len(my_mx[0])
        if any(len(row) != n for row in my_mx):
            return False
    for row in my_mx:
        for value in row:
            if not isinstance(value, (int, float)):
                return False
    return True


print(check_valid_mx(((1, 2), (3, 4))))
print(check_valid_mx(((1, 2), (3, 4, 5))))
print(check_valid_mx(((1, 2), [3, 4])))
print(check_valid_mx(((1, 2), (3, "4"))))
print(check_valid_mx([1, 2, 3]))

True
False
False
False
False


**说明**：依次检查外层类型、每个内层是否为元组、各行长度是否一致、元素是否为数值；任意一项不满足就返回 `False`。

### 4. mx_vec_mult

接受矩阵（元组的元组）和向量（元组），返回矩阵与向量相乘得到的数值元组。先用 `check_valid_mx` 检查矩阵，再检查向量类型和长度。

In [10]:
def mx_vec_mult(my_mx, vec):
    if not check_valid_mx(my_mx):
        raise TypeError("my_mx 不是有效矩阵")
    if not isinstance(vec, tuple):
        raise TypeError("vec 必须是元组")
    for value in vec:
        if not isinstance(value, (int, float)):
            raise TypeError("向量元素必须是 int 或 float")

    n = len(my_mx[0]) if my_mx else 0
    if len(vec) != n:
        raise ValueError("矩阵列数与向量长度不一致")

    result = []
    for row in my_mx:
        total = 0.0
        for j in range(n):
            total += row[j] * vec[j]
        result.append(total)
    return tuple(result)


print(mx_vec_mult(((1, 2), (3, 4)), (5, 6)))

try:
    mx_vec_mult(((1, 2), (3, 4, 5)), (1, 2))
except TypeError as e:
    print("TypeError:", e)

try:
    mx_vec_mult(((1, 2), (3, 4)), (1, 2, 3))
except ValueError as e:
    print("ValueError:", e)

(17.0, 39.0)
TypeError: my_mx 不是有效矩阵
ValueError: 矩阵列数与向量长度不一致


**说明**：结果向量的第 $i$ 个分量是矩阵第 $i$ 行与输入向量的内积。矩阵不合法时抛 `TypeError`，列数与向量长度不一致时抛 `ValueError`。

---
## 更多向量的乐趣

### 1. 定义 Vector 类

每个向量有一个维度（非负整数）和一个元素列表或元组。`Vector(dim, elements)`：只给维度时默认生成零向量；维度类型错误或为负数、维度与元素个数不一致时抛出异常。

In [11]:
class Vector:
    def __init__(self, dim, elements=None):
        if not isinstance(dim, int) or isinstance(dim, bool):
            raise TypeError("维度必须是整数")
        if dim < 0:
            raise ValueError("维度不能为负数")
        if elements is None:
            elements = [0] * dim
        if not isinstance(elements, (list, tuple)):
            raise TypeError("元素必须是列表或元组")
        if len(elements) != dim:
            raise ValueError("元素个数必须等于维度")
        for v in elements:
            if not isinstance(v, (int, float)):
                raise TypeError("元素必须是 int 或 float")
        self.dim = dim
        self.elements = tuple(elements)

    def _check_comparable(self, other):
        if not isinstance(other, Vector):
            return NotImplemented
        if self.dim != other.dim:
            raise ValueError("维度不同的向量不能比较")
        return other

    def __repr__(self):
        return f"Vector({self.dim}, {self.elements})"

    def __eq__(self, other):
        other = self._check_comparable(other)
        if other is NotImplemented:
            return NotImplemented
        return self.elements == other.elements

    def __lt__(self, other):
        other = self._check_comparable(other)
        if other is NotImplemented:
            return NotImplemented
        return self.elements < other.elements

    def __le__(self, other):
        other = self._check_comparable(other)
        if other is NotImplemented:
            return NotImplemented
        return self.elements <= other.elements

    def __gt__(self, other):
        other = self._check_comparable(other)
        if other is NotImplemented:
            return NotImplemented
        return self.elements > other.elements

    def __ge__(self, other):
        other = self._check_comparable(other)
        if other is NotImplemented:
            return NotImplemented
        return self.elements >= other.elements

    def dot(self, other):
        if not isinstance(other, Vector):
            raise TypeError("dot() 的参数必须是 Vector")
        if self.dim != other.dim:
            raise ValueError("维度不一致，无法计算内积")
        return float(sum(a * b for a, b in zip(self.elements, other.elements)))

    def __mul__(self, other):
        if isinstance(other, Vector):
            if self.dim != other.dim:
                raise ValueError("维度不一致，无法逐元素相乘")
            return Vector(self.dim, tuple(a * b for a, b in zip(self.elements, other.elements)))
        if isinstance(other, (int, float)):
            return Vector(self.dim, tuple(a * other for a in self.elements))
        return NotImplemented

    __rmul__ = __mul__

### 2. 类属性还是实例属性？

维度和元素应当是**实例属性**。每个 `Vector` 对象都有各自的维度和坐标，如果把 `dim`、`elements` 定义成类属性，所有实例会共享同一份数据，创建或修改一个向量会影响其他向量。这里在 `__init__` 中通过 `self.dim`、`self.elements` 把属性绑定到具体实例上。

### 3. 比较运算符

两个 `Vector` 坐标相同则相等；否则按元组的字典序比较（先比第一个坐标，再比第二个，依此类推）。比较维度不同的两个向量时抛出异常。

In [12]:
v1 = Vector(2, (2, 3))
v2 = Vector(2, (4, 5))

print("v1 =", v1)
print("v2 =", v2)
print("v1 == Vector(2, (2, 3)):", v1 == Vector(2, (2, 3)))
print("v1 < v2 :", v1 < v2)
print("v1 <= v2:", v1 <= v2)
print("v2 > v1 :", v2 > v1)
print("v2 >= v1:", v2 >= v1)

try:
    v1 == Vector(3, (1, 2, 3))
except ValueError as e:
    print("ValueError:", e)

v1 = Vector(2, (2, 3))
v2 = Vector(2, (4, 5))
v1 == Vector(2, (2, 3)): True
v1 < v2 : True
v1 <= v2: True
v2 > v1 : True
v2 >= v1: True
ValueError: 维度不同的向量不能比较


**说明**：相等性比较两个坐标元组是否完全相同；大小比较直接复用 Python 元组的字典序规则，因此 `(2, 3) < (4, 5)`。`_check_comparable` 统一检查类型和维度，维度不同时抛出 `ValueError`。

### 4. Vector.dot

接收另一个 `Vector`，返回两向量的内积；参数类型不正确或维度不一致时抛出异常。

In [13]:
print("v1.dot(v2) =", v1.dot(v2))

try:
    v1.dot((1, 2))
except TypeError as e:
    print("TypeError:", e)

try:
    v1.dot(Vector(3, (1, 2, 3)))
except ValueError as e:
    print("ValueError:", e)

v1.dot(v2) = 23.0
TypeError: dot() 的参数必须是 Vector
ValueError: 维度不一致，无法计算内积


**说明**：`v1.dot(v2)` 计算 $2\times4+3\times5=23.0$；非 `Vector` 参数抛 `TypeError`，维度不一致抛 `ValueError`。

### 5. 乘法运算符

标量左乘或右乘返回各元素乘以标量的新 `Vector`；两个 `Vector` 相乘时对应位置元素逐一相乘，结果仍是 `Vector`；维度不一致时抛出异常。

In [14]:
print("2 * v1 =", 2 * v1)
print("v1 * 2 =", v1 * 2)
print("v1 * v2 =", v1 * v2)
print("默认零向量 Vector(3) =", Vector(3))

try:
    v1 * Vector(3, (1, 2, 3))
except ValueError as e:
    print("ValueError:", e)

try:
    v1 * "x"
except TypeError as e:
    print("TypeError:", e)

2 * v1 = Vector(2, (4, 6))
v1 * 2 = Vector(2, (4, 6))
v1 * v2 = Vector(2, (8, 15))
默认零向量 Vector(3) = Vector(3, (0, 0, 0))
ValueError: 维度不一致，无法逐元素相乘
TypeError: can't multiply sequence by non-int of type 'Vector'


**说明**：`2 * v1` 会调用 `Vector.__rmul__`，`v1 * 2` 调用 `Vector.__mul__`，两者都返回新的 `Vector`。`v1 * v2` 是逐元素乘法，例如 `Vector(2, (2, 3)) * Vector(2, (4, 5))` 得到 `Vector(2, (8, 15))`。维度不同或操作数类型不支持时分别抛出 `ValueError` 和 `TypeError`。